# Customer Churn Prediction — Complete Analysis

**Team:** Samarth (Pipeline Lead) · Kartik (EDA Lead) · Hitesh (Modeling Lead)

This notebook contains the complete end-to-end analysis:
1. **Data Loading & Splitting** — 80/20 stratified split, leakage column removal
2. **Exploratory Data Analysis (EDA)** — Class balance, distributions, correlations, segment analysis
3. **Feature Engineering & Preprocessing Pipeline** — Custom transformers, ColumnTransformer, StandardScaler
4. **Model Training & Evaluation** — GridSearchCV for Logistic Regression, Random Forest, XGBoost
5. **Final Results** — Model comparison, confusion matrix, best model selection

---

## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import warnings

from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, ConfusionMatrixDisplay
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook', font_scale=1.1)

# Paths
ROOT = Path('..').resolve()
DATA = ROOT / 'data'
MODELS = ROOT / 'models'
REPORTS = ROOT / 'reports'
FIG_DIR = REPORTS / 'figures'
MODELS.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
PALETTE_CHURN = {0: '#3498db', 1: '#e74c3c'}

print('Setup complete.')

---
## 1. Data Loading & Train/Test Split

**Dataset:** Bank Customer Churn (Kaggle) — 10,000 customers, 18 features  
**Target:** `Exited` (1 = churned, 0 = stayed)  
**Leakage columns dropped:** `RowNumber`, `CustomerId`, `Surname`, `Complain`

In [ ]:
# Load raw dataset
raw = pd.read_csv(DATA / 'raw-bank-churn.csv')
print(f'Raw dataset: {raw.shape[0]} rows, {raw.shape[1]} columns')
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe()

In [ ]:
# Drop leakage / identifier columns
DROP_COLUMNS = ['RowNumber', 'CustomerId', 'Surname', 'Complain']
TARGET = 'Exited'

X = raw.drop(columns=DROP_COLUMNS + [TARGET])
y = raw[TARGET]

print(f'Features: {X.shape}')
print(f'\nTarget distribution:\n{y.value_counts()}')
print(f'\nChurn rate: {y.mean():.2%}')

In [ ]:
# Stratified 80/20 split (random_state=42 as required)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train churn rate: {y_train.mean():.2%}')
print(f'y_test  churn rate: {y_test.mean():.2%}')

In [ ]:
# Combine train features + target for EDA (NEVER use test set for EDA)
df = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
print(f'EDA DataFrame: {df.shape}')
df.head()

---
## 2. Exploratory Data Analysis (EDA)

> **All analysis below is on the TRAINING SET ONLY.** The test set is reserved for final evaluation.

### 2.1 Class Distribution

In [ ]:
counts = df['Exited'].value_counts().sort_index()
labels = ['Stayed (0)', 'Churned (1)']
colors = [PALETTE_CHURN[0], PALETTE_CHURN[1]]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
ax = axes[0]
bars = ax.bar(labels, counts.values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=13)
ax.set_ylabel('Count')
ax.set_title('Class Distribution of Exited (Churn)')
ax.set_ylim(0, counts.max() * 1.15)

# Pie chart
ax = axes[1]
ax.pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
       startangle=90, textprops={'fontsize': 12}, explode=(0, 0.05))
ax.set_title('Churn Proportion')

fig.tight_layout()
fig.savefig(FIG_DIR / '01_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

churn_rate = counts[1] / counts.sum()
print(f'\nChurn rate: {churn_rate:.2%} ({counts[1]:,} churned / {counts.sum():,} total)')
print('\n⚠️ INSIGHT: Only ~20% churned — the dataset is IMBALANCED.')
print('   → F1 score is the right metric. class_weight="balanced" should be used.')

### 2.2 Univariate Distributions (Numeric Features)

In [ ]:
NUMERIC_FEATURES = ['CreditScore', 'Age', 'Tenure', 'Balance',
                     'EstimatedSalary', 'Satisfaction Score', 'Point Earned']

n = len(NUMERIC_FEATURES)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_FEATURES):
    ax = axes[i]
    for label, color in PALETTE_CHURN.items():
        subset = df[df['Exited'] == label][col]
        ax.hist(subset, bins=30, alpha=0.5, color=color, density=True,
                label=f"{'Churned' if label else 'Stayed'}", edgecolor='white')
        subset.plot.kde(ax=ax, color=color, linewidth=2)
    ax.set_title(col)
    ax.set_xlabel('')
    ax.legend(fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Univariate Distributions — Numeric Features (by Churn)', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / '02_univariate_numeric.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n⚠️ INSIGHT: Age shows the strongest separation between churned vs stayed.')
print('   → Middle-aged customers (41-60) churn at much higher rates.')

### 2.3 Categorical Features vs Churn Rate

In [ ]:
CATEGORICAL_FEATURES = ['Geography', 'Gender', 'NumOfProducts',
                         'HasCrCard', 'IsActiveMember', 'Card Type']

for col in CATEGORICAL_FEATURES:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.3, 1]})

    # Count plot
    ax = axes[0]
    ct = df.groupby([col, 'Exited']).size().unstack(fill_value=0)
    ct.plot.bar(ax=ax, color=[PALETTE_CHURN[0], PALETTE_CHURN[1]], edgecolor='white')
    ax.set_title(f'{col} — Count by Churn Status')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend(['Stayed', 'Churned'], loc='upper right')
    ax.tick_params(axis='x', rotation=0)

    # Churn rate per category
    ax = axes[1]
    cr = df.groupby(col)['Exited'].mean().sort_values(ascending=False)
    bars = ax.barh(cr.index.astype(str), cr.values,
                   color=sns.color_palette('Set2', len(cr)), edgecolor='white')
    for bar, val in zip(bars, cr.values):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.1%}', va='center', fontweight='bold')
    ax.set_title(f'Churn Rate by {col}')
    ax.set_xlabel('Churn Rate')
    ax.set_xlim(0, min(cr.max() * 1.4, 1.0))

    fig.tight_layout()
    plt.show()

print('\n⚠️ KEY CATEGORICAL INSIGHTS:')
print('   → Germany has ~2x the churn rate of France/Spain')
print('   → Customers with 3-4 products churn at extreme rates (82-100%)')
print('   → Inactive members churn more (26.5%) vs active (14.6%)')

### 2.4 Correlation Analysis

In [ ]:
# Correlation heatmap
numeric_cols = NUMERIC_FEATURES + ['NumOfProducts', 'HasCrCard', 'IsActiveMember', 'Exited']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={'fontsize': 9})
ax.set_title('Correlation Heatmap — Numeric Features + Exited', fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / '09_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature correlations ranked by churn
churn_corr = corr['Exited'].drop('Exited').sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in churn_corr.values]
ax.barh(churn_corr.index, churn_corr.values, color=colors, edgecolor='white')
for i, (name, val) in enumerate(zip(churn_corr.index, churn_corr.values)):
    ax.text(val + (0.005 if val >= 0 else -0.005), i,
            f'{val:+.3f}', va='center',
            ha='left' if val >= 0 else 'right',
            fontweight='bold', fontsize=10)
ax.axvline(0, color='grey', linewidth=0.8)
ax.set_title('Feature Correlation with Churn (Exited)', fontsize=13)
ax.set_xlabel('Pearson Correlation')
fig.tight_layout()
fig.savefig(FIG_DIR / '10_churn_correlation_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n⚠️ INSIGHT: Age has the strongest positive correlation with churn.')
print('   → IsActiveMember has the strongest negative correlation (active = less churn).')

### 2.5 Segment Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 5a. Churn rate by tenure bucket
ax = axes[0, 0]
df['TenureBucket'] = pd.cut(df['Tenure'], bins=[-1, 1, 3, 5, 7, 10],
                             labels=['0-1', '2-3', '4-5', '6-7', '8-10'])
rate = df.groupby('TenureBucket', observed=True)['Exited'].mean()
ax.bar(rate.index.astype(str), rate.values, color=sns.color_palette('viridis', len(rate)), edgecolor='white')
for i, (cat, val) in enumerate(zip(rate.index, rate.values)):
    ax.text(i, val + 0.005, f'{val:.1%}', ha='center', fontweight='bold', fontsize=10)
ax.set_title('Churn Rate by Tenure Bucket')
ax.set_xlabel('Tenure (years)')
ax.set_ylabel('Churn Rate')
ax.set_ylim(0, rate.max() * 1.25)

# 5b. Churn rate by NumOfProducts
ax = axes[0, 1]
rate = df.groupby('NumOfProducts')['Exited'].agg(['mean', 'count'])
bars = ax.bar(rate.index.astype(str), rate['mean'],
              color=sns.color_palette('magma', len(rate)), edgecolor='white')
for i, (idx, row) in enumerate(rate.iterrows()):
    ax.text(i, row['mean'] + 0.01, f"{row['mean']:.1%}\n(n={row['count']:,})",
            ha='center', fontweight='bold', fontsize=9)
ax.set_title('Churn Rate by Number of Products')
ax.set_xlabel('NumOfProducts')
ax.set_ylabel('Churn Rate')
ax.set_ylim(0, min(rate['mean'].max() * 1.35, 1.05))

# 5c. Geography x Gender
ax = axes[0, 2]
cross = df.groupby(['Geography', 'Gender'])['Exited'].mean().unstack()
cross.plot.bar(ax=ax, color=[PALETTE_CHURN[1], PALETTE_CHURN[0]], edgecolor='white')
ax.set_title('Churn Rate: Geography × Gender')
ax.set_ylabel('Churn Rate')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Gender')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# 5d. Churn rate by age group
ax = axes[1, 0]
df['AgeGroup'] = pd.cut(df['Age'], bins=[17, 30, 40, 50, 60, 100],
                         labels=['18-30', '31-40', '41-50', '51-60', '60+'])
rate = df.groupby('AgeGroup', observed=True)['Exited'].mean()
ax.bar(rate.index.astype(str), rate.values, color=sns.color_palette('rocket', len(rate)), edgecolor='white')
for i, val in enumerate(rate.values):
    ax.text(i, val + 0.005, f'{val:.1%}', ha='center', fontweight='bold', fontsize=10)
ax.set_title('Churn Rate by Age Group')
ax.set_xlabel('Age Group')
ax.set_ylabel('Churn Rate')
ax.set_ylim(0, rate.max() * 1.2)

# 5e. Churn rate by balance bucket
ax = axes[1, 1]
df['BalanceBucket'] = pd.cut(df['Balance'],
                              bins=[-1, 0, 50_000, 100_000, 150_000, 300_000],
                              labels=['Zero', '1-50K', '50-100K', '100-150K', '150K+'])
rate = df.groupby('BalanceBucket', observed=True)['Exited'].mean()
ax.bar(rate.index.astype(str), rate.values, color=sns.color_palette('crest', len(rate)), edgecolor='white')
for i, val in enumerate(rate.values):
    ax.text(i, val + 0.005, f'{val:.1%}', ha='center', fontweight='bold', fontsize=10)
ax.set_title('Churn Rate by Balance Bucket')
ax.set_xlabel('Account Balance')
ax.set_ylabel('Churn Rate')
ax.set_ylim(0, rate.max() * 1.25)

# 5f. Churn rate by Satisfaction Score
ax = axes[1, 2]
rate = df.groupby('Satisfaction Score')['Exited'].mean()
ax.bar(rate.index.astype(str), rate.values, color=sns.color_palette('flare', len(rate)), edgecolor='white')
for i, val in enumerate(rate.values):
    ax.text(i, val + 0.005, f'{val:.1%}', ha='center', fontweight='bold', fontsize=10)
ax.set_title('Churn Rate by Satisfaction Score')
ax.set_xlabel('Satisfaction Score')
ax.set_ylabel('Churn Rate')
ax.set_ylim(0, rate.max() * 1.25)

fig.suptitle('Segment Analysis — Churn Rate by Customer Segments', fontsize=15, y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / '11_segment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Clean up temp columns
df.drop(columns=['TenureBucket', 'AgeGroup', 'BalanceBucket'], inplace=True)

### 2.6 EDA Summary & Key Insights

| # | Insight | Implication |
|---|---------|-------------|
| 1 | Only 20.4% churned — **class imbalance** | Use F1 score, `class_weight='balanced'` |
| 2 | **Age** is the strongest churn predictor (41-60 age group) | Create age buckets as engineered feature |
| 3 | **Germany** has ~2x the churn rate of France/Spain | Create `IsGermany` binary flag |
| 4 | Customers with **3-4 products** churn at 82-100% | NumOfProducts is critical |
| 5 | **Inactive members** churn more (26.5% vs 14.6%) | Create inactive × products interaction |
| 6 | **Non-zero balance** customers churn more (24.2% vs 13.5%) | Create `ZeroBalance` flag, `BalanceToSalaryRatio` |

---
## 3. Feature Engineering & Preprocessing Pipeline

Based on EDA insights, we engineer 6 new features and build a sklearn Pipeline.

### 3.1 Custom Feature Transformer

In [ ]:
class EDAFeatureEngineer(BaseEstimator, TransformerMixin):
    """Creates features based on EDA insights for churn prediction."""

    def __init__(self, age_bins=None, drop_originals=False):
        self.age_bins = age_bins or [0, 30, 40, 50, 60, 100]
        self.drop_originals = drop_originals

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)

        # AgeBucket — binned age (EDA: age is #1 churn predictor)
        if 'Age' in df.columns:
            df['AgeBucket'] = pd.cut(
                df['Age'], bins=self.age_bins,
                labels=range(len(self.age_bins) - 1),
                include_lowest=True
            ).astype(int)

        # BalanceToSalaryRatio — relative financial position
        if 'Balance' in df.columns and 'EstimatedSalary' in df.columns:
            df['BalanceToSalaryRatio'] = (
                df['Balance'] / df['EstimatedSalary'].clip(lower=1.0)
            ).clip(upper=10.0)

        # IsGermany — Germany has ~2x churn rate
        if 'Geography' in df.columns:
            df['IsGermany'] = (df['Geography'] == 'Germany').astype(int)

        # ZeroBalance — different churn pattern
        if 'Balance' in df.columns:
            df['ZeroBalance'] = (df['Balance'] == 0).astype(int)

        # InactiveProducts — inactive + multi-product = high risk
        if 'IsActiveMember' in df.columns and 'NumOfProducts' in df.columns:
            df['InactiveProducts'] = (1 - df['IsActiveMember']) * df['NumOfProducts']

        # HighValueAtRisk — composite flag for highest-risk segment
        has_age = 'Age' in df.columns
        has_act = 'IsActiveMember' in df.columns
        has_geo = 'Geography' in df.columns
        has_bal = 'Balance' in df.columns
        if has_age and has_act and (has_geo or has_bal):
            old = df['Age'] >= 40
            inactive = df['IsActiveMember'] == 0
            de = (df['Geography'] == 'Germany') if has_geo else False
            hi_bal = (df['Balance'] > 100_000) if has_bal else False
            df['HighValueAtRisk'] = (old & inactive & (de | hi_bal)).astype(int)

        return df

print('EDAFeatureEngineer defined.')
print('\nEngineered features:')
print('  1. AgeBucket         — ordinal age bins (5 groups)')
print('  2. BalanceToSalaryRatio — relative financial position')
print('  3. IsGermany         — binary flag for Germany')
print('  4. ZeroBalance       — zero vs non-zero balance')
print('  5. InactiveProducts  — inactive × num products interaction')
print('  6. HighValueAtRisk   — composite high-risk flag')

### 3.2 Build Preprocessing Pipeline

In [ ]:
# Column groups after feature engineering
CAT_COLS = ['Geography', 'Gender', 'Card Type']
NUM_COLS = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary',
            'Satisfaction Score', 'Point Earned', 'NumOfProducts',
            'AgeBucket', 'BalanceToSalaryRatio', 'InactiveProducts']
BIN_COLS = ['HasCrCard', 'IsActiveMember', 'IsGermany', 'ZeroBalance', 'HighValueAtRisk']

# ColumnTransformer
ct = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), CAT_COLS),
        ('num', StandardScaler(), NUM_COLS),
        ('pass', 'passthrough', BIN_COLS),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

# Full pipeline: feature engineering → column transformer
pipe = Pipeline([
    ('feat_eng', EDAFeatureEngineer()),
    ('col_trans', ct),
])

print('Pipeline structure:')
print(pipe)

In [ ]:
# Fit on train, transform both
X_train_processed = pipe.fit_transform(X_train)
X_test_processed = pipe.transform(X_test)

# Get feature names
try:
    feat_names = list(ct.get_feature_names_out())
except Exception:
    ohe = ct.named_transformers_['cat']
    feat_names = []
    for feat, cats in zip(CAT_COLS, ohe.categories_):
        for c in cats[1:]:
            feat_names.append(f'{feat}_{c}')
    feat_names.extend(NUM_COLS)
    feat_names.extend(BIN_COLS)

X_train_proc_df = pd.DataFrame(X_train_processed, columns=feat_names)
X_test_proc_df = pd.DataFrame(X_test_processed, columns=feat_names)

print(f'X_train processed: {X_train_proc_df.shape}')
print(f'X_test  processed: {X_test_proc_df.shape}')
print(f'\nFeatures ({len(feat_names)}): {feat_names}')
print(f'\nNaN check — train: {X_train_proc_df.isna().sum().sum()}, test: {X_test_proc_df.isna().sum().sum()}')

X_train_proc_df.head()

In [ ]:
# Save pipeline artifact
joblib.dump(pipe, MODELS / 'preprocessing_pipeline.joblib')
print(f'Pipeline saved to {MODELS / "preprocessing_pipeline.joblib"}')

# Save processed data
X_train_proc_df.to_csv(DATA / 'X_train_processed.csv', index=False)
X_test_proc_df.to_csv(DATA / 'X_test_processed.csv', index=False)
print('Processed data saved.')

---
## 4. Model Training & Evaluation

We train three models using **GridSearchCV** (5-fold CV, F1-scored) and evaluate on the held-out test set.

**Evaluation metric:** `f1_score(y_test, model.predict(X_test))` — using default 0.5 threshold as required.

In [ ]:
# Prepare data
y_train_arr = y_train.values.ravel() if hasattr(y_train, 'values') else y_train
y_test_arr = y_test.values.ravel() if hasattr(y_test, 'values') else y_test

# Class balance check
vals, cnts = np.unique(y_train_arr, return_counts=True)
print('Class balance in y_train:')
for v, c in zip(vals, cnts):
    print(f'  class {v}: {c} ({c / len(y_train_arr):.1%})')

In [ ]:
# Define models and hyperparameter grids
models = {
    'LogisticRegression': (
        LogisticRegression(random_state=RANDOM_STATE, max_iter=2000, class_weight='balanced'),
        {'C': [0.01, 0.1, 1, 10], 'penalty': ['l2'], 'solver': ['lbfgs']}
    ),
    'RandomForest': (
        RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
        {'n_estimators': [200, 400], 'max_depth': [5, 10, None], 'min_samples_leaf': [1, 3, 5]}
    ),
    'XGBoost': (
        XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss'),
        {'n_estimators': [200, 400], 'max_depth': [3, 5, 7],
         'learning_rate': [0.05, 0.1], 'scale_pos_weight': [1, 3]}
    ),
}

print(f'{len(models)} models configured for training.')

In [ ]:
# Train and evaluate each model
results = []
fitted_models = {}

for name, (estimator, param_grid) in models.items():
    print(f"\n{'=' * 60}")
    print(f'Training {name}')
    print('=' * 60)

    grid = GridSearchCV(estimator, param_grid, scoring='f1', cv=5, n_jobs=-1, verbose=1)
    grid.fit(X_train_processed, y_train_arr)

    best_model = grid.best_estimator_
    fitted_models[name] = best_model

    # Evaluate on test set (default threshold, predict() not predict_proba())
    y_pred = best_model.predict(X_test_processed)
    test_f1 = f1_score(y_test_arr, y_pred)

    print(f'  Best params: {grid.best_params_}')
    print(f'  Best CV F1 (train-only): {grid.best_score_:.4f}')
    print(f'  Test F1 (held-out): {test_f1:.4f}')
    print(classification_report(y_test_arr, y_pred))

    results.append({
        'model': name,
        'best_params': grid.best_params_,
        'cv_f1_train': grid.best_score_,
        'test_f1': test_f1,
    })

---
## 5. Final Results

In [ ]:
# Model comparison table
comparison_df = pd.DataFrame(results).sort_values('test_f1', ascending=False)

print('=' * 60)
print('MODEL COMPARISON (sorted by test F1)')
print('=' * 60)
print(comparison_df[['model', 'cv_f1_train', 'test_f1']].to_string(index=False))

comparison_df

In [ ]:
# Visualize model comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison_df))
width = 0.35

bars1 = ax.bar(x - width/2, comparison_df['cv_f1_train'], width, label='CV F1 (Train)', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + width/2, comparison_df['test_f1'], width, label='Test F1', color='#e74c3c', edgecolor='white')

ax.set_xlabel('Model')
ax.set_ylabel('F1 Score')
ax.set_title('Model Comparison — CV F1 vs Test F1')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['model'])
ax.legend()
ax.set_ylim(0, 0.8)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontweight='bold', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontweight='bold', fontsize=10)

fig.tight_layout()
plt.show()

In [ ]:
# Best model — confusion matrix
best_name = comparison_df.iloc[0]['model']
best_model = fitted_models[best_name]
best_f1 = comparison_df.iloc[0]['test_f1']

y_pred_best = best_model.predict(X_test_processed)
cm = confusion_matrix(y_test_arr, y_pred_best)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Stayed', 'Churned'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'Confusion Matrix — {best_name} (Test F1 = {best_f1:.4f})', fontsize=13)
fig.tight_layout()
plt.show()

print(f'\n🏆 BEST MODEL: {best_name} (test F1 = {best_f1:.4f})')
print(f'\nClassification Report:')
print(classification_report(y_test_arr, y_pred_best, target_names=['Stayed', 'Churned']))

In [ ]:
# Save best model and comparison table
joblib.dump(best_model, MODELS / 'final_model.joblib')
print(f'Best model saved to {MODELS / "final_model.joblib"}')

comparison_df.to_csv(REPORTS / 'model_comparison.csv', index=False)
print(f'Comparison table saved to {REPORTS / "model_comparison.csv"}')

print('\n' + '=' * 60)
print('✅ ANALYSIS COMPLETE')
print('=' * 60)
print(f'  Best model: {best_name}')
print(f'  Test F1:    {best_f1:.4f}')
print(f'  Dashboard:  https://customer-churn-prediction-i.streamlit.app/')